In [ ]:
import albumentations as A
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import os
import segmentation_models as sm
import tensorflow as tf
import tensorflow.keras.backend as K
import wandb
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam
from wandb.keras import WandbCallback

from networks import *

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
HEIGHT = 240
WIDTH = 320
CHANNELS = 3
IMAGE_SHAPE = (HEIGHT, WIDTH, CHANNELS)
BATCH_SIZE = 4
EPOCHS = 50
STEPS_PER_EPOCH = 1500
LEARNING_RATE = 3e-4
MOMENTUM = 0.5
DROPOUT = 0.0
DEEP_SUPERVISION = False
LOGGING = False

config = {
    'model': 'UNet',
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'steps_per_epoch': STEPS_PER_EPOCH,
    'loss': 'categorical_crossentropy',
    'optimizer': 'Adam',
    'learning_rate': LEARNING_RATE,
    'momentum': MOMENTUM,
    'dropout': DROPOUT,
    'image_shape': IMAGE_SHAPE,
    'deep_supervision': DEEP_SUPERVISION,
}

In [ ]:
if LOGGING:
    wandb.login()
    wandb.init(project='PVID-assignment-4', config=config)

## Load dataset

In [ ]:
# Load dataset
dataset_path = 'input/'
train_ds, val_ds, test_ds = load_dataset(
    dataset_path, val_split=0.1, batch_size=BATCH_SIZE,
    height=HEIGHT, width=WIDTH, channels=CHANNELS,
    preprocess=None, augment=True, repeat=True,
)

In [ ]:
for code, name, weight in zip(CLASS_TO_COLOR.keys(), CLASS_NAMES, CLASS_WEIGHTS):
    print(f'Class {code}: {name} [weight = {weight}]')
print()

try:
    print('Number of batches in training set:', len(train_ds))
except TypeError:
    print('Training set is repeated indefinitely.')
print('Number of batches in validation set:', len(val_ds))
print('Number of batches in test set:', len(test_ds))
print()

data_batch = next(iter(val_ds))
images, masks = data_batch
print('Image batch shape:', images.shape)
print('Mask batch shape:', masks.shape)

reshaped = np.reshape(masks, (-1, 3))
mask_pixels = np.unique(reshaped, axis=0)
print('Unique mask pixels:\n', mask_pixels)

# Show some iamge-mask pairs for sanity check
for i, (img, mask) in enumerate(zip(images, masks)):
    if i > 4: break
    visualize_pair(img, mask, mask_type='one-hot')

## Training

In [ ]:
# SM losses
sm_dice = sm.losses.DiceLoss(class_weights=CLASS_WEIGHTS)
sm_focal = sm.losses.CategoricalFocalLoss()
sm_cce = sm.losses.CategoricalCELoss(class_weights=CLASS_WEIGHTS)
sm_jac = sm.losses.JaccardLoss(class_weights=CLASS_WEIGHTS)
# sm_total_loss = (1. * sm_dice) + (1. * sm_focal) + (1. * sm_cce) + (1. * sm_jac)
sm_total_loss = sm.losses.categorical_focal_dice_loss

# SM metrics
sm_iou = sm.metrics.IOUScore(threshold=0.5)
sm_f1 = sm.metrics.FScore(threshold=0.5)

In [ ]:
model = UNet(
    input_shape=IMAGE_SHAPE,
    num_classes=NUM_CLASSES,
    num_filters=32,
    dropout=DROPOUT,
    batchnorm=True,
    final_activation='softmax',
    # deep_supervision=DEEP_SUPERVISION,
)

# model = SuimNet(
#     input_shape=IMAGE_SHAPE,
#     num_classes=NUM_CLASSES,
# )

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE, beta_1=MOMENTUM),
    loss=sm_total_loss,
    metrics=['accuracy', jaccard_coef, dice_coef, sm_iou, sm_f1],
)

tf.keras.utils.plot_model(model, show_shapes=True)
model.summary()

In [ ]:
# Smaller dataset for testing
EPOCHS = 3
STEPS_PER_EPOCH = 2
train_ds = train_ds.take(STEPS_PER_EPOCH)
val_ds = val_ds.take(STEPS_PER_EPOCH)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, verbose=1),
    VisualizeProgress(
        test_data=next(iter(val_ds)), every_n_epochs=1, num_samples=5,
        convert_mask=one_hot_to_rgb, deep_supervision=DEEP_SUPERVISION, log=LOGGING,
    ),
]
if LOGGING:
    callbacks.append(WandbCallback())

history = model.fit(
    train_ds, validation_data=val_ds, validation_steps=len(val_ds),
    epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks, verbose=1,
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 8))

plt.subplot(221)
plt.title('Loss')
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()

plt.subplot(222)
plt.title('Accuracy')
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.legend()

plt.subplot(223)
plt.title('Jaccard')
plt.plot(history.history['jaccard_coef'], label='train')
plt.plot(history.history['val_jaccard_coef'], label='val')
plt.legend()

plt.subplot(224)
plt.title('Dice')
plt.plot(history.history['dice_coef'], label='train')
plt.plot(history.history['val_dice_coef'], label='val')
plt.legend()

plt.tight_layout()
plt.savefig('training-history.png')
plt.show()

In [ ]:
# save model
model.save('models/basic-unet-model.h5')

In [ ]:
if LOGGING:
    wandb.finish()

## Evaluation

In [ ]:
# load model
model = tf.keras.models.load_model('models/basic-unet-model.h5', custom_objects={
    'jaccard_coef_loss': lambda y_true, y_pred: -jaccard_coef(y_true, y_pred),
    'jaccard_coef': jaccard_coef,
    'dice_coef': dice_coef,
})

In [ ]:
# load model for testing
model = tf.keras.models.load_model('models/sm-unet-model.h5', compile=False)

In [ ]:
test = next(iter(test_ds))
model.evaluate(test[0], test[1])

In [ ]:
test = next(iter(test_ds))

predictions = model.predict(test[0])
predictions.shape